# EEG 信号处理完整流程

从原始 GDF 数据到分类评估的完整 pipeline
包括：预处理 → 特征提取 → 分类 → 评估

字体和图片显示设置


In [ ]:
import code._plot_config
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# 添加项目路径以便导入模块
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'code'))

from code.config import DEFAULT_CONFIG, TASK_CLASS_NAMES, epochs_events_to_class_labels
from classification.svm_classifier import train_eeg_svm_pipeline, plot_confusion_matrix
from pretreatment.complete_preprocessing import complete_preprocessing_pipeline
import numpy as np

print(f"项目根目录: {project_root}")
print("✅ 所有模块导入成功")


## Step 1: 数据准备

In [ ]:
subject_id = "A01T"
data_path = project_root / "BCICIV_2a_gdf" / f"{subject_id}.gdf"
print(f"被试 ID: {subject_id}")
print(f"数据文件: {data_path}")
print(f"文件存在: {data_path.exists()}")


## Step 2: EEG 预处理

In [ ]:
epochs, ica = complete_preprocessing_pipeline(subject=subject_id)
print(f"✅ 预处理完成")
print(f"   - Epochs 形状：{epochs.get_data().shape}")
print(f"   - 试次数：{len(epochs)}")
print(f"   - 通道数：{len(epochs.ch_names)}")


## Step 3: 准备标签

使用 `epochs_events_to_class_labels` 动态映射 MNE 内部事件 ID → 类别标签 (1-4)

In [ ]:
y = epochs_events_to_class_labels(epochs)
unique_classes, class_counts = np.unique(y, return_counts=True)
print("标签分布:")
for cls, cnt in zip(unique_classes, class_counts):
    cls_name = TASK_CLASS_NAMES[cls - 1]
    print(f"   类别 {cls} ({cls_name}): {cnt} 个")
print(f"标签形状: {y.shape}")


## Step 4: 防泄漏 SVM Pipeline 训练与评估

使用 `train_eeg_svm_pipeline` 将特征提取、标准化、SVM 封装在 sklearn Pipeline 中，
交叉验证时每折独立拟合特征提取器，**彻底避免数据泄漏**。

In [ ]:
cfg = DEFAULT_CONFIG
results = {}

# 4.1 CSP 特征
print("--- CSP 特征 ---")
clf_csp, cv_csp, acc_csp = train_eeg_svm_pipeline(
    epochs, y, feature_set="csp",
    cv_folds=cfg.cv_folds, n_csp_components=cfg.csp_components,
    kernel=cfg.svm_kernel, random_state=cfg.random_state)
results["CSP"] = {"clf": clf_csp, "cv": cv_csp, "acc": acc_csp}

# 4.2 小波特征
print("\n--- 小波特征 ---")
clf_wav, cv_wav, acc_wav = train_eeg_svm_pipeline(
    epochs, y, feature_set="wavelet",
    cv_folds=cfg.cv_folds, wavelet=cfg.wavelet, wavelet_level=cfg.wavelet_level,
    kernel=cfg.svm_kernel, random_state=cfg.random_state)
results["Wavelet"] = {"clf": clf_wav, "cv": cv_wav, "acc": acc_wav}

# 4.3 融合特征（全通道）
print("\n--- 融合特征（全通道） ---")
clf_fused, cv_fused, acc_fused = train_eeg_svm_pipeline(
    epochs, y, feature_set="fused",
    cv_folds=cfg.cv_folds, n_csp_components=cfg.csp_components,
    wavelet=cfg.wavelet, wavelet_level=cfg.wavelet_level,
    motor_channels_only=False, kernel=cfg.svm_kernel, random_state=cfg.random_state)
results["Fused"] = {"clf": clf_fused, "cv": cv_fused, "acc": acc_fused}

# 4.4 融合特征（运动区 C3/Cz/C4）
print("\n--- 融合特征（运动区） ---")
clf_fm, cv_fm, acc_fm = train_eeg_svm_pipeline(
    epochs, y, feature_set="fused",
    cv_folds=cfg.cv_folds, n_csp_components=cfg.csp_components,
    wavelet=cfg.wavelet, wavelet_level=cfg.wavelet_level,
    motor_channels_only=True, kernel=cfg.svm_kernel, random_state=cfg.random_state)
results["Fused-Motor"] = {"clf": clf_fm, "cv": cv_fm, "acc": acc_fm}

# 4.5 FBCSP
print("\n--- FBCSP ---")
clf_fb, cv_fb, acc_fb = train_eeg_svm_pipeline(
    epochs, y, feature_set="fb_csp",
    cv_folds=cfg.cv_folds, n_csp_components=cfg.csp_components,
    freq_bands=[(8,12),(12,16),(16,20),(20,24),(24,30)],
    kernel=cfg.svm_kernel, random_state=cfg.random_state)
results["FBCSP"] = {"clf": clf_fb, "cv": cv_fb, "acc": acc_fb}


## Step 5: 结果汇总

In [ ]:
print("=" * 60)
print("分类性能对比")
print("=" * 60)
for name, r in results.items():
    print(f"{name:14s} 准确率: {r['acc']:.4f} ± {r['cv'].std():.4f}")
print("=" * 60)

best_name = max(results, key=lambda k: results[k]["acc"])
print(f"\n🏆 最佳方法: {best_name} (准确率: {results[best_name]['acc']:.4f})")


## Step 6: 混淆矩阵可视化

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix

best_clf = results[best_name]["clf"]
X = epochs.get_data()
cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.random_state)
y_pred = cross_val_predict(best_clf, X, y, cv=cv)
cm = confusion_matrix(y, y_pred, labels=[1,2,3,4])
plot_confusion_matrix(cm, class_names=TASK_CLASS_NAMES,
                      save_path=f"./{subject_id}_confusion_matrix.png")
print("✅ 混淆矩阵已保存（基于交叉验证预测，无数据泄漏）")

## 完成

In [ ]:
print("=" * 80)
print(f"✅ 被试 {subject_id} 完整流程处理完成！")
print(f"📈 试次数: {len(epochs)}, 通道数: {len(epochs.ch_names)}")
print(f"🏆 最佳方法: {best_name} ({results[best_name]['acc']:.4f})")
print("=" * 80)
